In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from download_data import get_clinical_patient_data, get_clinical_sample_data


In [ ]:
study_id = "luad_mskcc_2023_met_organotropism"
clinical_patient= get_clinical_patient_data(study_id)
clinical_sample = get_clinical_sample_data(study_id)



[4/4] Downloading patient-level clinical data...
   ✓ 2298 patients × 33 columns

[3/4] Downloading sample-level clinical data...
   ✓ 2653 samples × 60 columns


In [ ]:
clinical_sample['patientId'] = clinical_sample['sampleId'].str.extract(r'(P-\d+)') # get the patient id from sample id
df = pd.merge(clinical_patient, clinical_sample, how='inner', on='patientId')
df = df[df['GROUP_NO'].isin(['Group1', 'Group2'])]
percent_missing_per_feature = df.isnull().mean().sort_values(ascending=False)
drop_cols = percent_missing_per_feature[percent_missing_per_feature > .2].index.to_list()
df = df.drop(columns=drop_cols)

# drop rows with missing values
df = df.dropna(axis=0)
# drop rows where race is unknown
df_clean = df[df['RACE'] != 'Unknown']


In [135]:
target_cols = ['EVER_MET_SITE_ADRENAL', 
               'EVER_MET_SITE_BONE', 
               'EVER_MET_SITE_CNS',
               'EVER_MET_SITE_LIVER_BILIARY_TRACT', 
               'EVER_MET_SITE_LN',
               'EVER_MET_SITE_LUNG', 
               'EVER_MET_SITE_PLEURA']


# remove metadata other than patientId, sampleId, and GENE_PANEL
metadata_cols = [
    'GROUP_NO',
    'CANCER_TYPE', 'CANCER_TYPE_DETAILED', 'ONCOTREE_CODE',
    'PRIMARY_SITE', 'SAMPLE_CLASS', 'SAMPLE_COVERAGE',
    'SAMPLE_COUNT', 'INSTITUTE', 'IN_MATCHED',
    'IMPACT_PRIMARY_GROUP', 'SOMATIC_STATUS'
]

leak_cols = [
    'ADRENAL_STATUS', 'BONE_STATUS', 'CNS_STATUS',
    'LIVER_STATUS', 'LN_STATUS', 'LUNG_STATUS', 'PLEURA_STATUS',
    # Future outcome info
    'DEATH', 'OS_MONTHS', 'OS_STATUS', 'FU_2YRS', 'METASTATIC_BURDEN',
    # Post sample treatment
    'POST_SAMPLE_CHEMOTHERAPY', 'POST_SAMPLE_IMMUNOTHERAPY',
    'POST_SAMPLE_TARGETED', 'POST_SAMPLE_TX', 'POST_SAMPLE_XRT',
    # Adjuvant (post surgery)
    'ADJUVANT', 'ADJUVANT_CHEMOTHERAPY', 'ADJUVANT_IMMUNOTHERAPY',
    'ADJUVANT_TARGETED', 'ADJUVANT_XRT', 'ADJUVANT_THERAPY',
]

# keep only neoadjuvant feature, which is yes if any of the following are true, and no if all are false
multicollinearity = ['NEOADJUVANT_CHEMOTHERAPY', 'NEOADJUVANT_IMMUNOTHERAPY',
    'NEOADJUVANT_TARGETED', 'NEOADJUVANT_XRT']

genomic_cols = [
    'FGA', 'FRACTION_GENOME_ALTERED', 'IS_WGD',
    'MSI_SCORE', 'MSI_TYPE', 'MUTATION_COUNT',
    'PLOIDY', 'PURITY', 'TMB_NONSYNONYMOUS',
    'CELL_CYCLE', 'HIPPO', 'MYC_PATH', 'NOTCH',
    'NRF2', 'PI3K', 'RTK_RAS', 'TGF_BETA', 'TP53_PATH', 'WNT'
]

# Drop all columns that are metadata, can cause potential leakage, and exclude genomic features as well. Separate the target from the features
y = df_clean[target_cols]
X = df_clean.drop(columns=target_cols + metadata_cols + leak_cols + multicollinearity + genomic_cols)

# Drop any column with one unique value, does not contribute to model
X = X.loc[:, X.nunique() > 1] 


In [136]:
# encode features using one hot encoding
one_hot = ['CIGARETTE_HX', 'NEOADJUVANT', 'RACE', 'SEX',
       'PREDOM_HISTO_SUBTYPE', 'PRE_SAMPLE_CHEMOTHERAPY',
       'PRE_SAMPLE_IMMUNOTHERAPY', 'PRE_SAMPLE_TARGETED', 'PRE_SAMPLE_TX',
       'PRE_SAMPLE_XRT']
X_ENCODED= pd.get_dummies(X, columns=one_hot, drop_first=True)

# clinical stage and pathological stage's ordinal values may have clinical meaning. 
# We could go one hot encoding as well,
# but we choose to preserve the ordinal information
X_ENCODED['CSTAGE'] = X_ENCODED['CSTAGE'].map({'Stage I': 1, 'Stage II': 2, 'Stage III': 3, 'Stage IV': 4})
X_ENCODED['PSTAGE'] = X_ENCODED['PSTAGE'].map({'Stage I': 1, 'Stage II': 2, 'Stage III': 3})


In [137]:
X_ENCODED


,patientId,sampleId,AGE_AT_DOS_BX,CSTAGE,GENE_PANEL,PSTAGE,CIGARETTE_HX_Never,NEOADJUVANT_Yes,RACE_Black,RACE_Other,...,PREDOM_HISTO_SUBTYPE_Lepidic,PREDOM_HISTO_SUBTYPE_Micropapillary,PREDOM_HISTO_SUBTYPE_Papillary,PREDOM_HISTO_SUBTYPE_Solid,PREDOM_HISTO_SUBTYPE_Unknown,PRE_SAMPLE_CHEMOTHERAPY_Yes,PRE_SAMPLE_IMMUNOTHERAPY_Yes,PRE_SAMPLE_TARGETED_Yes,PRE_SAMPLE_TX_Yes,PRE_SAMPLE_XRT_Yes
4,P-0000219,P-0000219-T01-IM3,52.91780822,1,IMPACT341,3,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
7,P-0000348,P-0000348-T01-IM3,65.08767123,3,IMPACT341,3,True,True,False,False,...,False,False,True,False,False,True,False,False,True,False
37,P-0000867,P-0000867-T01-IM3,64.94794521,1,IMPACT341,2,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
61,P-0001241,P-0001241-T01-IM3,83.17808219,1,IMPACT341,2,True,False,False,False,...,False,False,False,False,True,False,False,False,False,False
88,P-0001828,P-0001828-T01-IM3,75.00821918,2,IMPACT341,1,False,True,False,False,...,False,False,False,False,True,True,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2034,P-0040055,P-0040055-T01-IM6,62.76438356,1,IMPACT468,2,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2041,P-0040473,P-0040473-T02-IM6,70.59726027,1,IMPACT468,1,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
2046,P-0040655,P-0040655-T01-IM6,51.29315068,3,IMPACT468,2,False,True,False,False,...,False,False,False,False,False,True,False,False,True,True
2066,P-0041217,P-0041217-T01-IM6,71.74794521,2,IMPACT468,3,True,False,False,False,...,False,False,True,False,False,False,False,False,False,False


In [138]:
#save to csv
y.to_csv('../data/processed/y.csv', index=False)
X_ENCODED.to_csv('../data/processed/X_clinical.csv', index=False)
